## Music Recommendation
### University of Virginia
### DS 5110: Big Data Systems
### Last Updated: March 18, 2026


---

#### Instructions

In this assignment, you will build an ALS recommendation algorithm based on user listening data from Autoscrobbler.

The data consists of: 
- user data (listeners)
- item data (songs)
- interaction data (user listens, which is implicit feedback).  

Additional details are in the `README` file. Be sure to review in advance.

Develop/test code to accomplish the tasks below, clearly showing the results and documenting your work.

You will use the **DataFrame API** for all work.

**TOTAL POINTS: 10**
***

#### Preprocessing Steps in order:

1) To reduce runtime, you will sample 20% of the interaction data from `user_artist_data.txt` using seed=314 (and use this seed throughout the lab)
2) Next, remove users that listened to fewer than 10 **artists** (you may ignore aliasing in this step)
3) Lastly, split the data: 60% for training / 20% for validation / 20% for testing

In [12]:
# ----------------------------------------------------------------------------------------------------------------------------------------------
# PREPROCESSING STEP 1
# To reduce runtime, you will sample 20% of the interaction data from user_artist_data.txt using seed=314 (and use this seed throughout the lab)
# ----------------------------------------------------------------------------------------------------------------------------------------------

# Before doing anything, create a Spark session.
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

# Define the schema for the file to import.  
# -- I added this after reviewing the data file and running an initial import.
# -- Even though I saw the file had no headers, I didn't initially stop to think I needed to add them!  :)
from pyspark.sql.types import StructType, StructField, IntegerType
autoscrobbler_schema = StructType([
                                    StructField("userid", IntegerType()),
                                    StructField("artistid", IntegerType()),
                                    StructField("playcount", IntegerType()),
                                    ])
     

# Load the data.
# The columns are separated by a space, so need to include sep=" ".
df_user = spark.read.csv("/standard/ds7200-apt4c/large_datasets/user_artist_data.txt",  sep=" ", header=False, schema=autoscrobbler_schema)

# Check to see if any columns have null values.
df_user.createOrReplaceTempView("df_user")
df_user_null = spark.sql("""
    SELECT COUNT(*) AS RowsWithNulls
    FROM df_user
    WHERE 
        userid IS NULL 
        OR artistid IS NULL
        OR playcount IS NULL
""")
print(f"""There are {df_user_null.show()} rows with NULL values.""")

# Since no rows have null values, continue...
# Select a 20% sample from the data.
df_user_small = df_user.sample(fraction=0.20, seed=314)

# Confirm rows were imported.
print(f"""There are {df_user_small.count()} rows in the imported data set.""")

# Review a few records and the schema to make sure it was applied correctly.
# And it does, so yay!
df_user.show(5)
df_user.printSchema()

+-------------+
|RowsWithNulls|
+-------------+
|            0|
+-------------+

There are None rows with NULL values.
There are 4859650 rows in the imported data set.
+-------+--------+---------+
| userid|artistid|playcount|
+-------+--------+---------+
|1000002|       1|       55|
|1000002| 1000006|       33|
|1000002| 1000007|        8|
|1000002| 1000009|      144|
|1000002| 1000010|      314|
+-------+--------+---------+
only showing top 5 rows
root
 |-- userid: integer (nullable = true)
 |-- artistid: integer (nullable = true)
 |-- playcount: integer (nullable = true)



In [13]:
# ----------------------------------------------------------------------------------------------------------------------------------------------
# PREPROCESSING STEP 2
# Next, remove users that listened to fewer than 10 artists (you may ignore aliasing in this step)
# ----------------------------------------------------------------------------------------------------------------------------------------------

# Remove users that listen to fewer than 10 artists using SQL.
# -- First register the DF as a temp view first.
df_user_small.createOrReplaceTempView("df_user_small")

# See how may users have listened to fewer than 10 artists.

# Select the columns needed for this analysis.
df_user_go = spark.sql("""
    SELECT *
    FROM df_user_small
    WHERE userid IN
        (
            SELECT userid
            FROM df_user_small
            GROUP BY userid
            HAVING COUNT(DISTINCT artistid) > 9
        )
""")

print(f"""There are {df_user_go.count()} rows after removing users that listened to less than 10 artists.""")

[Stage 10275:>                                                      (0 + 8) / 8]

There are 4684468 rows after removing users that listened to less than 10 artists.


In [14]:
# ----------------------------------------------------------------------------------------------------------------------------------------------
# PREPROCESSING STEP 3
# Lastly, split the data: 60% for training / 20% for validation / 20% for testing
# ----------------------------------------------------------------------------------------------------------------------------------------------
# Split the dataset.
trainDF, valDF, testDF = df_user_go.randomSplit([.6, .2, .2], seed=314)

# Cache the datasets for better downstream processing.
trainDF = trainDF.cache()
valDF = valDF.cache()
testDF = testDF.cache()

print(f"""There are {trainDF.count()} rows in the training set, {valDF.count()} rows in the validation set, and {testDF.count()} in the test set.""")

26/10/11 16:32:42 WARN CacheManager: Asked to cache already cached data.
26/10/11 16:32:42 WARN CacheManager: Asked to cache already cached data.
26/10/11 16:32:42 WARN CacheManager: Asked to cache already cached data.


There are 2808949 rows in the training set, 938515 rows in the validation set, and 937004 in the test set.


In [15]:
# TASK 1 (1 PT)
# Show the first row from the training set
trainDF.show(1)

# Result should remain consistent with seed=314.
#  |userid|artistid|playcount|
#  +------+--------+---------+
#  |   321|      40|        1|
#  +------+--------+---------+

+------+--------+---------+
|userid|artistid|playcount|
+------+--------+---------+
|   536|      30|        5|
+------+--------+---------+
only showing top 1 row


In [16]:
# TASK 2 (1 PT)
# Show the first row from the test set. We will refer to this user as Test User 1.
testDF.show(1)

# Result should remain consistent with seed=314.
#  +------+--------+---------+
#  |userid|artistid|playcount|
#  +------+--------+---------+
#  |   321|     357|        3|
#  +------+--------+---------+

+------+--------+---------+
|userid|artistid|playcount|
+------+--------+---------+
|   536|     341|        1|
+------+--------+---------+
only showing top 1 row


In [31]:
# ALS MODEL 1: Train the ALS implicit model on the training data (since the measurements are activity and not ratings)
# Use parameters: rank=8, maxIter=5, regParam=0.02, alpha=0.01

# The parameters above do not align to the old MLlib library, so using the newer ML library.
from pyspark.ml.recommendation import ALS

# Create the model specification using the parameters for ML.
als_spec = ALS(
    userCol="userid",
    itemCol="artistid",
    ratingCol="playcount",
    implicitPrefs=True,
    coldStartStrategy="drop",
    rank=8,
    maxIter=5,
    regParam=0.02,
    alpha=0.01
)

# Fit the model.
als_model = als_spec.fit(trainDF)

print("The model has been trained.")

The model has been trained.


In [32]:
# TASK 3 (2 PTS)
# Using ALS MODEL 1, make 10 recommendations for Test User 1
als_predictions = als_model.transform(testDF)

# Use SQL to get 10 predictions for Test User 1 (userid = 321).
als_predictions.createOrReplaceTempView("als_predictions")

# STEP 1: First, get the predictions for userid = 321.
test_user1 = spark.sql("""
    SELECT *
    FROM als_predictions
    WHERE 
        userid=321
        AND NOT ISNAN(prediction)
""")

# STEP 2: Then, get the top 10 predictions.
test_user1.createOrReplaceTempView("test_user1")
test_user1_top10 = spark.sql("""
    SELECT *
    FROM test_user1
    ORDER BY prediction DESC
    LIMIT 10
""")


test_user1_top10.show()

+------+--------+---------+-----------+
|userid|artistid|playcount| prediction|
+------+--------+---------+-----------+
|   321| 1275996|        1| 0.11322815|
|   321|    2178|        3| 0.09492492|
|   321|    1991|       12| 0.08725764|
|   321|    4951|      262| 0.05056441|
|   321|     930|        1| 0.04292053|
|   321|     304|       19| 0.03267776|
|   321| 1028974|        3|0.030446978|
|   321|    1365|        8|0.023087572|
|   321|     378|        7|0.021886308|
|   321| 1233802|        7| 0.02055233|
+------+--------+---------+-----------+



**TASK 4 (1 PT)**  
**Briefly explain Recall@K in the context of recommender systems**

Recall@k is like the concept of recall in class imbalance problems which is the number of True Positives (TP) that are correctly predicted, e.g., TP / (TP + FN).  What's different for recommendation problems is that the equivalent of True Positives (the numerator) is the top number (k) of the total predicted items of which the user did interact with and which the model correctly predicted as such, and the denominator is this number plus all the other items that the user interacted with.

In [33]:
# TASK 5 (1 PT)
# Calculate Recall@K for Test User 1

# This is asking, "Of all the artists that the user listened to, how many of the top-k predictions did the model correctly predict?"
# Mathematically, the formula is:
#
#              # artists the user listened to that are predicted in the top k
#  Recall@K = ----------------------------------------------------------------
#                     # all of the artists the user listened to


# One way to do this is with SQL.
# I'll stick with k=10 for this.

# This requires, querying the test data.
testDF.createOrReplaceTempView("testDF")

# STEP 1: DENOMINATOR.
# Get the total rows for userid = 321.
all_items = spark.sql("""
SELECT *
FROM testDF
WHERE userid = 321
""")

# STEP 2: NUMERATOR.
# Get the total of these that were in the predicted set; these are the true positives (tp).
# This part uses the top k=10 from TASK 3.
all_items.createOrReplaceTempView("all_items")
test_user1_top10.createOrReplaceTempView("test_user1_top10")
tp = spark.sql("""
SELECT *
FROM test_user1_top10 AS t
JOIN all_items AS a
ON t.artistid = a.artistid
""")

# STEP 3: CALCULATE the Recall@K.
tp.createOrReplaceTempView("tp")
recall_k = spark.sql("""
SELECT 
    ROUND(
            (SELECT COUNT(*) FROM tp) /
            (SELECT COUNT(*) FROM all_items)
            ,4
        ) AS RecallK
""")

# I had trouble extracting the value from the recall_k DataFrame so I had MS Copilot give me this syntax.
# I just don't do enough straight up Python coding for indexing to have sunk in.
print(f"The Recall@K for userid=321 is: {recall_k.collect()[0]['RecallK']}")

The Recall@K for userid=321 is: 0.2703


In [20]:
# ALS MODEL 2: For this model, you will use hyperparameter tuning on the validation set to select the best values for
# (rank, maxIter, regParam, alpha) to maximize Recall@K

# Import the necessary library.
from pyspark.ml.tuning import ParamGridBuilder

# Build the hyperparameter tuning grid.
# I had Gemini help recommend values for these parameters.
# I added comments to explain my understanding of why to use these values.
paramGrid = (
        ParamGridBuilder()
    
        # Specify different number of hidden features to look for.
        .addGrid(als_spec.rank, [10, 20, 40])
    
        # Specify how many times to loop through the data.
        .addGrid(als_spec.maxIter, [10])
    
        # Specify the regularization penalty; 0.01 is flexible and 0.1 is stricter (less-overfitting).
        .addGrid(als_spec.regParam, [0.01, 0.1])
    
        # Specify how much weight is placed on the playcounts data; 1.0 a little weight, 10.0 a lot of weight.
        .addGrid(als_spec.alpha, [1.0, 10.0])
    
        # Build the tuning grid.
        .build()
    )

print("The paramGrid has been built.")

The paramGrid has been built.


In [34]:
# TASK 6 (2 PTS)
# Report the results including best values for (rank, maxIter, regParam, alpha) along with the Recall@K from the validation set

# Import the necessary libraries.
from pyspark.ml.tuning import CrossValidator
from pyspark.ml.evaluation import RegressionEvaluator

# This evaluator will evaluate RMSE to tune parameters and then Recall@K will be calculated later on the validation set.
evaluator_spec = RegressionEvaluator(
    predictionCol="prediction",  # Use the ALS predictions.
    labelCol="playcount",        # Use the playcount data as the "truth" from the held-out data set for each CV fold.
    metricName="rmse",           # Specifies to optimize RMSE.
)

# Specify the cross validation approach.
crossval = CrossValidator(
    estimator=als_spec,
    estimatorParamMaps=paramGrid,
    evaluator= evaluator_spec,
    numFolds=2
)

print("The evaluator and crossval specs are built.")

The evaluator and crossval specs are built.


In [35]:
# THIS STEP TAKES SOME TIME SO IT IS PUT INTO ITS OWN CELL.

# Time this step.
import time
t0 = time.time()

# Run cross-validation on the validation data, leveraging parallel processing.
cvModel_val = crossval.setParallelism(4).fit(valDF)

# Show how much elapsed.
print("train time:", time.time() - t0)

train time: 32.07768702507019


In [40]:
# Extract the best performing model.
best_model_val = cvModel_val.bestModel

# Find the index of the best performing run.
best_idx = cvModel_val.avgMetrics.index(min(cvModel_val.avgMetrics))

# Pull the exact parameter map used for that best run.
best_params = cvModel_val.getEstimatorParamMaps()[best_idx]

# Print the resulting values.
print("--- Best Hyperparameters Found ---")
for param, value in best_params.items(): print(f"Best {param.name}: {value}")

print("\n--- Best RMSE ---")
print(f"Best Validation RMSE: {min(cvModel_val.avgMetrics):.4f}")

--- Best Hyperparameters Found ---
Best rank: 10
Best maxIter: 10
Best regParam: 0.01
Best alpha: 1.0

--- Best RMSE ---
Best Validation RMSE: 128.6288


In [ ]:
# Retrain the best configuration on train + validation data.


In [ ]:
# TASK 7 (1 PT)
# Calculate Recall@K on the test set and report the value.

In [ ]:
# TASK 8 (1 PT)
# Calculate Recall@K for Test User 1 
# Justify the result, showing the recommendations and user activity. Did hyperparameter tuning seem to help for this user?
